# 05 — Softmax Selection

**Goal:** Convert adjusted template scores into a **probability distribution** using softmax, and select templates by sampling.

This notebook covers:
- The exploration vs exploitation dilemma
- The softmax (Boltzmann) formula
- How the temperature parameter τ controls the balance
- A full single-event walkthrough: scores → recency → softmax → selection

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_sample, parse_history, parse_eligible_templates
from src.scoring.difference_score import (
    compute_template_reward_rates,
    compute_relative_difference_scores_fast,
)
from src.scoring.bayesian_smoothing import bayesian_smooth
from src.recency.recency_penalty import adjust_scores_with_recency
from src.bandit.softmax_selector import (
    softmax_probabilities,
    softmax_select,
    explain_softmax_selection,
)

sns.set_style("whitegrid")
print("Ready!")

---
## 1. Recompute Smoothed Scores

We need the full pipeline up to Bayesian smoothing as our starting point.

In [ ]:
train_df = load_sample(n_rows=100_000, split="train")
reward_rates, counts = compute_template_reward_rates(train_df)
rds_scores = compute_relative_difference_scores_fast(train_df, reward_rates)
smoothed_scores = bayesian_smooth(rds_scores, counts, kappa=1000)

templates = sorted(smoothed_scores.keys())
print(f"\nSmoothed scores ready for {len(templates)} templates.")

---
## 2. The Exploration-Exploitation Dilemma

After scoring, the simplest strategy is **greedy**: always pick the highest-scoring template. But this has a problem:

- What if our scores are slightly wrong?
- What if a template we've barely tried is actually the best?
- We'd never discover it because we keep picking the current leader.

The solution: assign **probabilities** to each template based on scores, then **sample**. Good templates are more likely, but others still get tried occasionally.

---
## 3. The Softmax Formula

$$P(a) = \frac{e^{\tau \times \text{score}(a)}}{\sum_{t \in \text{eligible}} e^{\tau \times \text{score}(t)}}$$

The **temperature parameter τ** controls the balance:

| τ | Behavior |
|---|----------|
| τ → 0 | Uniform random (pure exploration) |
| τ = 10 | Slight preference for best |
| τ = 50 | Strong preference for best |
| τ → ∞ | Always picks the best (greedy / pure exploitation) |

In [ ]:
# Use first 5 templates for clarity
demo_templates = templates[:5]
demo_scores = {t: smoothed_scores[t] for t in demo_templates}

print("Demo scores:")
for t in demo_templates:
    print(f"  {t}: {demo_scores[t]:+.6f}")

---
## 4. How τ Changes the Probabilities

In [ ]:
tau_values = [1, 10, 50, 100]

print(f"{'τ':<6}", end="")
for t in demo_templates:
    print(f"{t:>10}", end="")
print(f"{'Max P':>10}")
print("-" * (6 + 10 * len(demo_templates) + 10))

for tau in tau_values:
    probs = softmax_probabilities(demo_templates, demo_scores, tau)
    print(f"{tau:<6}", end="")
    for t in demo_templates:
        print(f"{probs[t]:>10.4f}", end="")
    print(f"{max(probs.values()):>10.4f}")

In [ ]:
# Grouped bar chart
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(demo_templates))
width = 0.18
colors = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c"]

for i, tau in enumerate(tau_values):
    probs = softmax_probabilities(demo_templates, demo_scores, tau)
    prob_vals = [probs[t] for t in demo_templates]
    ax.bar(x + i * width, prob_vals, width, label=f"τ={tau}", color=colors[i], edgecolor="white")

ax.axhline(y=1/len(demo_templates), color="gray", linestyle=":", alpha=0.7,
           label=f"Uniform = {1/len(demo_templates):.3f}")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(demo_templates)
ax.set_title("Softmax Probabilities at Different Temperatures", fontsize=14, fontweight="bold")
ax.set_xlabel("Template")
ax.set_ylabel("Selection Probability")
ax.legend()
plt.tight_layout()
plt.show()

print("Low τ → nearly uniform (explore everything).")
print("High τ → concentrates on the best (exploit the leader).")

---
## 5. Simulation — 1000 Draws at τ=50

Let's sample 1,000 times and see if the empirical distribution matches the theoretical probabilities.

In [ ]:
tau = 50
rng = np.random.default_rng(42)

selections = [softmax_select(demo_templates, demo_scores, tau, rng) for _ in range(1000)]
theoretical = softmax_probabilities(demo_templates, demo_scores, tau)

print(f"{'Template':<10} {'Theoretical':>12} {'Empirical':>12} {'Draws':>8}")
print("-" * 44)
for t in demo_templates:
    emp = selections.count(t) / 1000
    print(f"{t:<10} {theoretical[t]:>12.4f} {emp:>12.4f} {selections.count(t):>8}")

In [ ]:
# Side-by-side bar chart
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(demo_templates))
width = 0.35

theo_vals = [theoretical[t] for t in demo_templates]
emp_vals = [selections.count(t) / 1000 for t in demo_templates]

ax.bar(x - width/2, theo_vals, width, label="Theoretical", color="#3498db", edgecolor="white")
ax.bar(x + width/2, emp_vals, width, label="Empirical (1000 draws)", color="#e67e22", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(demo_templates)
ax.set_title(f"Theoretical vs Empirical Selection (τ={tau})", fontsize=14, fontweight="bold")
ax.set_ylabel("Probability")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Full Single-Event Walkthrough

Let's trace one real event through the entire pipeline: **smoothed scores → recency adjustment → softmax → selection**.

In [ ]:
# Find a row with non-empty history for a more interesting demo
example_idx = None
for i in range(len(train_df)):
    h = parse_history(train_df["history"].iloc[i])
    elig = train_df["eligible_templates"].iloc[i]
    if isinstance(elig, str):
        import ast
        elig = ast.literal_eval(elig)
    if len(h) >= 2 and len(elig) >= 4:
        example_idx = i
        break

if example_idx is None:
    example_idx = 0  # fallback

row = train_df.iloc[example_idx]
eligible = row["eligible_templates"]
if isinstance(eligible, str):
    eligible = ast.literal_eval(eligible)
history = parse_history(row["history"])

print(f"Event #{example_idx}")
print(f"  Selected:  {row['selected_template']}")
print(f"  Reward:    {row['session_end_completed']}")
print(f"  Eligible:  {eligible}")
print(f"  History:   {history}")

In [ ]:
# Step 1: Get smoothed scores for eligible templates
print("STEP 1: Smoothed scores (from training)")
eligible_scores = {t: smoothed_scores.get(t, 0.0) for t in eligible}
for t in eligible:
    print(f"  {t}: {eligible_scores[t]:+.6f}")

# Step 2: Apply recency penalty
gamma, h_val = 0.1, 0.5
print(f"\nSTEP 2: Apply recency penalty (γ={gamma}, h={h_val})")
adjusted = adjust_scores_with_recency(eligible_scores, history, gamma, h_val)
for t in eligible:
    diff = adjusted[t] - eligible_scores[t]
    marker = f" (penalty={abs(diff):.6f})" if abs(diff) > 0.00001 else ""
    print(f"  {t}: {eligible_scores[t]:+.6f} → {adjusted[t]:+.6f}{marker}")

# Step 3: Softmax probabilities
tau = 50
print(f"\nSTEP 3: Softmax probabilities (τ={tau})")
probs = explain_softmax_selection(eligible, adjusted, tau)

# Step 4: Sample
print(f"\nSTEP 4: Sample a template")
rng = np.random.default_rng(42)
selected = softmax_select(eligible, adjusted, tau, rng)
print(f"  → Selected: {selected} (probability was {probs[selected]:.4f})")
print(f"  (The actual data chose: {row['selected_template']})")

---
## 7. Summary

| Concept | What it does |
|---------|-------------|
| **Softmax** | Converts scores into probabilities using $P(a) \propto e^{\tau \times \text{score}(a)}$ |
| **Temperature (τ)** | Low = explore, High = exploit |
| **Selection** | Sample from the probability distribution |

At this point we have the **complete decision pipeline**: raw rates → RDS → Bayesian smoothing → recency penalty → softmax → selection.

The final question: **does it actually work?** We need to evaluate our policy against the random baseline using offline evaluation.

**Next notebook:** `06_full_pipeline_evaluation.ipynb` — fit the policy, evaluate with importance sampling, measure lift.